# PPO + LSTM for ViZDoom

An agent that completes Doom levels using **Proximal Policy Optimization** with **LSTM** memory.

- **CNN** extracts spatial features from stacked grayscale frames
- **LSTM** carries temporal context across steps
- **Actor-Critic** shares the CNN+LSTM trunk
- **GAE** for advantage estimation

References: [PPO paper](https://arxiv.org/abs/1707.06347) · [ViZDoom](https://vizdoom.farama.org/) · [Spinning Up](https://spinningup.openai.com/en/latest/algorithms/ppo.html)

## 0. Install dependencies

In [1]:
%pip install gymnasium
%pip install vizdoom
%pip install opencv-python

Note: you may need to restart the kernel to use updated packages.

Note: you may need to restart the kernel to use updated packages.


## 1. Imports

In [2]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import multiprocessing as mp
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.distributions import Categorical
import cv2
import vizdoom as vzd

# doom_worker.py must be in the same directory — imported by worker processes via spawn
from doom_worker import env_worker

torch.backends.cudnn.benchmark = True

print(f"PyTorch: {torch.__version__}")
print(f"Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")
print(f"CPUs available: {mp.cpu_count()}")

PyTorch: 2.5.1+cu121
Device: cuda
CPUs available: 24


## 2. Hyperparameters

In [ ]:
# ── Network ─────────────────────────────────────────────────────────────────────
HIDDEN_DIM   = 512
LSTM_HIDDEN  = 256

# ── PPO ─────────────────────────────────────────────────────────────────────────
LR            = 2.5e-4
GAMMA         = 0.99
GAE_LAMBDA    = 0.95
PPO_CLIP      = 0.2
PPO_EPOCHS    = 4
MINI_BATCH    = 256   # 6144/256 = 24 mini-batches per epoch
ENTROPY_COEF  = 0.05
VALUE_COEF    = 0.5
MAX_GRAD_NORM = 0.5

# ── Training ────────────────────────────────────────────────────────────────────
FRAME_STACK   = 4
IMG_H, IMG_W  = 84, 84
ROLLOUT_STEPS = 512   # steps per env per rollout → 512×12=6144 samples/update
TOTAL_STEPS   = 2_000_000
N_ENVS        = 12    # 12/24 threads, ~700MB VRAM per update

## 3. ViZDoom Environment

Wrapper that handles:
- loading the scenario `.cfg`
- frame preprocessing (resize + normalization)
- frame stacking (last `FRAME_STACK` frames as policy input)

In [4]:
# Minimal actions for basic: enemy is always in front, no turning needed
BASIC_BUTTONS = [
    vzd.Button.MOVE_LEFT,
    vzd.Button.MOVE_RIGHT,
    vzd.Button.ATTACK,
]

# Full actions for defend_the_center and deadly_corridor
FULL_BUTTONS = [
    vzd.Button.MOVE_FORWARD,
    vzd.Button.MOVE_BACKWARD,
    vzd.Button.MOVE_LEFT,
    vzd.Button.MOVE_RIGHT,
    vzd.Button.TURN_LEFT,
    vzd.Button.TURN_RIGHT,
    vzd.Button.ATTACK,
]

def make_env(scenario="basic", buttons=FULL_BUTTONS):
    game = vzd.DoomGame()
    game.load_config(f"{vzd.scenarios_path}/{scenario}.cfg")
    game.set_screen_format(vzd.ScreenFormat.GRAY8)
    game.set_screen_resolution(vzd.ScreenResolution.RES_160X120)
    game.set_window_visible(False)
    game.clear_available_buttons()
    for btn in buttons:
        game.add_available_button(btn)
    game.init()
    return game


def preprocess_frame(frame):
    frame = cv2.resize(frame, (IMG_W, IMG_H), interpolation=cv2.INTER_AREA)
    return frame.astype(np.float32) / 255.0


class DoomEnv:
    """Thin wrapper around VizDoom with frame stacking."""

    def __init__(self, scenario="basic", buttons=FULL_BUTTONS):
        self.game        = make_env(scenario, buttons)
        self.num_actions = len(buttons)
        self.frame_buffer = np.zeros((FRAME_STACK, IMG_H, IMG_W), dtype=np.float32)
        self.actions     = np.eye(self.num_actions, dtype=np.uint8).tolist()

    def reset(self):
        self.game.new_episode()
        frame = preprocess_frame(self.game.get_state().screen_buffer)
        self.frame_buffer = np.stack([frame] * FRAME_STACK, axis=0)
        return self.frame_buffer.copy()

    def step(self, action_idx):
        reward = self.game.make_action(self.actions[action_idx])
        done   = self.game.is_episode_finished()
        if done:
            next_frame = np.zeros((IMG_H, IMG_W), dtype=np.float32)
        else:
            next_frame = preprocess_frame(self.game.get_state().screen_buffer)
        self.frame_buffer = np.roll(self.frame_buffer, -1, axis=0)
        self.frame_buffer[-1] = next_frame
        return self.frame_buffer.copy(), reward, done

    def close(self):
        self.game.close()


print("Basic actions:", [b.name for b in BASIC_BUTTONS])
print("Full actions: ", [b.name for b in FULL_BUTTONS])

Basic actions: ['MOVE_LEFT', 'MOVE_RIGHT', 'ATTACK']
Full actions:  ['MOVE_FORWARD', 'MOVE_BACKWARD', 'MOVE_LEFT', 'MOVE_RIGHT', 'TURN_LEFT', 'TURN_RIGHT', 'ATTACK']


In [ ]:
class VecDoomEnv:
    """
    N parallel ViZDoom environments, each running in its own subprocess.
    Uses multiprocessing 'spawn' (required on Windows).
    Workers are defined in doom_worker.py so they can be imported by spawned processes.
    """

    def __init__(self, n_envs, scenario, buttons):
        self.n_envs  = n_envs
        self.pipes   = []
        self.workers = []
        ctx = mp.get_context('spawn')
        for _ in range(n_envs):
            parent_conn, child_conn = ctx.Pipe()
            p = ctx.Process(
                target=env_worker,
                args=(child_conn, scenario, buttons),
                daemon=True,
            )
            p.start()
            child_conn.close()
            self.pipes.append(parent_conn)
            self.workers.append(p)
        print(f"Started {n_envs} workers for '{scenario}'")

    def reset(self):
        for conn in self.pipes:
            conn.send(('reset', None))
        return np.array([conn.recv() for conn in self.pipes])

    def step(self, actions):
        for conn, a in zip(self.pipes, actions):
            conn.send(('step', int(a)))
        results = [conn.recv() for conn in self.pipes]
        states  = np.array([r[0] for r in results], dtype=np.float32)
        rewards = np.array([r[1] for r in results], dtype=np.float32)
        dones   = np.array([r[2] for r in results], dtype=bool)
        return states, rewards, dones

    def close(self):
        # Terminate directly — avoids ViZDoom cleanup deadlock on Windows
        for w in self.workers:
            if w.is_alive():
                w.terminate()
        for w in self.workers:
            w.join(timeout=3)
        for conn in self.pipes:
            try:
                conn.close()
            except Exception:
                pass

## 4. Actor-Critic Network (CNN + LSTM)

```
Input (4×84×84)
    └─► CNN ──► Linear(3136→512)
                    └─► LSTMCell(512→256)
                              ├─► Actor  Linear(256→N_actions) → logits
                              └─► Critic Linear(256→1)         → V(s)
```

In [6]:
class ActorCriticLSTM(nn.Module):

    def __init__(self, num_actions):
        super().__init__()

        self.cnn = nn.Sequential(
            nn.Conv2d(FRAME_STACK, 32, kernel_size=8, stride=4),  # → 32×20×20
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=4, stride=2),           # → 64×9×9
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, stride=1),           # → 64×7×7
            nn.ReLU(),
            nn.Flatten(),
            nn.Linear(64 * 7 * 7, HIDDEN_DIM),
            nn.ReLU(),
        )

        self.lstm   = nn.LSTMCell(HIDDEN_DIM, LSTM_HIDDEN)
        self.actor  = nn.Linear(LSTM_HIDDEN, num_actions)
        self.critic = nn.Linear(LSTM_HIDDEN, 1)

    def forward(self, x, lstm_state):
        """
        x          : (batch, FRAME_STACK, H, W)
        lstm_state : (h, c)  each (batch, LSTM_HIDDEN)
        returns    : logits, value, (h, c)
        """
        features = self.cnn(x)
        h, c     = self.lstm(features, lstm_state)
        logits   = self.actor(h)
        value    = self.critic(h)
        return logits, value, (h, c)

    def init_lstm_state(self, batch_size=1):
        device = next(self.parameters()).device
        return (
            torch.zeros(batch_size, LSTM_HIDDEN, device=device),
            torch.zeros(batch_size, LSTM_HIDDEN, device=device),
        )


# Quick shape check (using FULL_BUTTONS as reference)
dummy_policy = ActorCriticLSTM(len(FULL_BUTTONS))
dummy_x      = torch.zeros(1, FRAME_STACK, IMG_H, IMG_W)
dummy_state  = dummy_policy.init_lstm_state()
logits, val, _ = dummy_policy(dummy_x, dummy_state)
print(f"Full model  — logits: {logits.shape}  |  params: {sum(p.numel() for p in dummy_policy.parameters()):,}")
dummy_basic = ActorCriticLSTM(len(BASIC_BUTTONS))
logits_b, _, _ = dummy_basic(dummy_x, dummy_basic.init_lstm_state())
print(f"Basic model — logits: {logits_b.shape}  |  params: {sum(p.numel() for p in dummy_basic.parameters()):,}")

Full model  — logits: torch.Size([1, 7])  |  params: 2,474,664
Basic model — logits: torch.Size([1, 3])  |  params: 2,473,636


## 5. Rollout Buffer

In [7]:
class RolloutBuffer:
    """
    Stores one rollout across N parallel environments.
    Each list entry is one timestep snapshot across all envs:
        states   : (N, FRAME_STACK, H, W)
        actions  : (N,)
        logprobs : (N,)
        rewards  : (N,)
        dones    : (N,)
        values   : (N,)
        lstm_h/c : (N, LSTM_HIDDEN)
    """

    def __init__(self):
        self.states   = []
        self.actions  = []
        self.logprobs = []
        self.rewards  = []
        self.dones    = []
        self.values   = []
        self.lstm_h   = []
        self.lstm_c   = []

    def clear(self):
        self.__init__()

## 6. GAE (Generalized Advantage Estimation)

$$A_t^{\text{GAE}} = \sum_{l=0}^{\infty} (\gamma\lambda)^l \delta_{t+l}, \quad \delta_t = r_t + \gamma V(s_{t+1}) - V(s_t)$$

In [8]:
def compute_gae(rewards, values, dones, last_values):
    """
    Multi-env GAE.
    rewards, values, dones : (T, N) numpy arrays
    last_values            : (N,)  bootstrapped V(s_{T+1}) for each env
    Returns advantages and returns, both flattened to (T*N,).
    """
    T, N       = rewards.shape
    advantages = np.zeros_like(rewards)
    gae        = np.zeros(N, dtype=np.float32)

    for t in reversed(range(T)):
        next_val      = last_values if t == T - 1 else values[t + 1]
        next_nonterminal = 1.0 - dones[t]
        delta         = rewards[t] + GAMMA * next_val * next_nonterminal - values[t]
        gae           = delta + GAMMA * GAE_LAMBDA * next_nonterminal * gae
        advantages[t] = gae

    returns = advantages + values
    # Flatten row-major: (T, N) -> (T*N,)
    return advantages.reshape(-1), returns.reshape(-1)

## 7. PPO Update

$$L^{\text{CLIP}} = \mathbb{E}\left[\min\left(r_t A_t,\ \text{clip}(r_t, 1-\varepsilon, 1+\varepsilon) A_t\right)\right]$$

$$L = L^{\text{CLIP}} - c_v L^{\text{VF}} + c_e H$$

In [9]:
def ppo_update(policy, optimizer, buffer, last_values,
               entropy_coef=ENTROPY_COEF, ppo_epochs=PPO_EPOCHS):
    """
    PPO update over a multi-env rollout buffer.
    buffer contains T timesteps × N envs of experience.
    last_values: (N,) bootstrapped critic values at the end of the rollout.
    """
    device = next(policy.parameters()).device

    # Stack to (T, N) arrays
    rewards = np.array(buffer.rewards,  dtype=np.float32)   # (T, N)
    dones   = np.array(buffer.dones,    dtype=np.float32)   # (T, N)
    values  = np.array(buffer.values,   dtype=np.float32)   # (T, N)

    advantages, returns = compute_gae(rewards, values, dones, last_values)
    advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)

    T, N = rewards.shape
    # Flatten all arrays to (T*N, ...)
    states_arr   = np.array(buffer.states,   dtype=np.float32).reshape(T * N, FRAME_STACK, IMG_H, IMG_W)
    actions_arr  = np.array(buffer.actions,  dtype=np.int64).reshape(T * N)
    logprobs_arr = np.array(buffer.logprobs, dtype=np.float32).reshape(T * N)
    lstm_h_arr   = np.array(buffer.lstm_h,   dtype=np.float32).reshape(T * N, LSTM_HIDDEN)
    lstm_c_arr   = np.array(buffer.lstm_c,   dtype=np.float32).reshape(T * N, LSTM_HIDDEN)

    states_t   = torch.tensor(states_arr,   dtype=torch.float32).to(device)
    actions_t  = torch.tensor(actions_arr,  dtype=torch.long).to(device)
    logprobs_t = torch.tensor(logprobs_arr, dtype=torch.float32).to(device)
    adv_t      = torch.tensor(advantages,   dtype=torch.float32).to(device)
    returns_t  = torch.tensor(returns,      dtype=torch.float32).to(device)
    lstm_h_t   = torch.tensor(lstm_h_arr,   dtype=torch.float32).to(device)
    lstm_c_t   = torch.tensor(lstm_c_arr,   dtype=torch.float32).to(device)

    indices = np.arange(T * N)

    for _ in range(ppo_epochs):
        np.random.shuffle(indices)
        for start in range(0, T * N, MINI_BATCH):
            idx = indices[start:start + MINI_BATCH]
            if len(idx) < 2:
                continue

            logits, value, _ = policy(states_t[idx], (lstm_h_t[idx], lstm_c_t[idx]))
            dist      = Categorical(logits=logits)
            logprobs  = dist.log_prob(actions_t[idx])
            entropy   = dist.entropy().mean()

            ratio  = torch.exp(logprobs - logprobs_t[idx])
            surr1  = ratio * adv_t[idx]
            surr2  = torch.clamp(ratio, 1 - PPO_CLIP, 1 + PPO_CLIP) * adv_t[idx]

            actor_loss  = -torch.min(surr1, surr2).mean()
            critic_loss = F.mse_loss(value.squeeze(-1), returns_t[idx])
            loss        = actor_loss + VALUE_COEF * critic_loss - entropy_coef * entropy

            optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(policy.parameters(), MAX_GRAD_NORM)
            optimizer.step()

## 8. Training Loop

In [ ]:
def train(scenario="basic", total_steps=TOTAL_STEPS, save_path="ppo_lstm_doom.pth",
          buttons=FULL_BUTTONS, n_envs=N_ENVS,
          early_stop_reward=None, early_stop_window=100,
          entropy_coef=ENTROPY_COEF, ppo_epochs=PPO_EPOCHS,
          rollback_threshold=1.0):
    """
    Multi-env PPO training from scratch.
    rollback_threshold : rollback only when avg < best * threshold.
                         1.0 = any decline triggers rollback.
                         0.75 = tolerate up to 25% drop before rolling back.
    """
    from tqdm.notebook import tqdm

    num_actions  = len(buttons)
    device       = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    ckpt_base    = save_path.replace(".pth", "")
    effective_lr = LR

    print(f"Training on {device}  |  scenario: {scenario}  |  n_envs: {n_envs}  |  actions: {num_actions}  |  steps: {total_steps:,}")
    print(f"entropy: {entropy_coef}  |  ppo_epochs: {ppo_epochs}  |  lr: {effective_lr:.2e}  |  batch/update: {ROLLOUT_STEPS * n_envs:,}")
    if early_stop_reward is not None:
        print(f"Early stop: avg{early_stop_window} >= {early_stop_reward}")

    vec_env = VecDoomEnv(n_envs, scenario, buttons)
    policy  = ActorCriticLSTM(num_actions).to(device)
    opt     = optim.Adam(policy.parameters(), lr=effective_lr)
    buffer  = RolloutBuffer()

    states = vec_env.reset()
    lstm_h = torch.zeros(n_envs, LSTM_HIDDEN, device=device)
    lstm_c = torch.zeros(n_envs, LSTM_HIDDEN, device=device)

    ep_rewards      = np.zeros(n_envs, dtype=np.float32)
    reward_history  = []
    episode         = 0
    step            = 0

    milestones_done = set()
    milestones      = {int(total_steps * p / 100): p for p in range(10, 101, 10)}
    last_ckpt_avg   = -float('inf')
    last_ckpt_path  = None
    rollbacks       = 0

    pbar = tqdm(total=total_steps, unit="step", dynamic_ncols=True)

    while step < total_steps:

        for _ in range(ROLLOUT_STEPS):
            states_t = torch.tensor(states, dtype=torch.float32).to(device)

            with torch.inference_mode():
                logits, values, (new_h, new_c) = policy(states_t, (lstm_h, lstm_c))
                dist     = Categorical(logits=logits)
                actions  = dist.sample()
                logprobs = dist.log_prob(actions)

            next_states, rewards, dones = vec_env.step(actions.cpu().numpy())

            buffer.states.append(states.copy())
            buffer.actions.append(actions.cpu().numpy())
            buffer.logprobs.append(logprobs.cpu().numpy())
            buffer.rewards.append(rewards)
            buffer.dones.append(dones.astype(np.float32))
            buffer.values.append(values.squeeze(-1).cpu().numpy())
            buffer.lstm_h.append(lstm_h.cpu().numpy())
            buffer.lstm_c.append(lstm_c.cpu().numpy())

            done_mask = torch.tensor(dones, dtype=torch.float32, device=device).unsqueeze(1)
            lstm_h = new_h * (1.0 - done_mask)
            lstm_c = new_c * (1.0 - done_mask)

            ep_rewards += rewards
            for i, done in enumerate(dones):
                if done:
                    reward_history.append(float(ep_rewards[i]))
                    ep_rewards[i] = 0.0
                    episode += 1

            states  = next_states
            step   += n_envs
            pbar.update(n_envs)

        if reward_history:
            avg = np.mean(reward_history[-early_stop_window:])
            pbar.set_postfix(ep=episode, avg=f"{avg:.1f}", rb=rollbacks)

            if (early_stop_reward is not None
                    and len(reward_history) >= early_stop_window
                    and avg >= early_stop_reward):
                ckpt_path = f"{ckpt_base}_ckptES.pth"
                torch.save(policy.state_dict(), ckpt_path)
                pbar.write(
                    f"[EARLY STOP] step {step:,} | ep {episode} "
                    f"| avg{early_stop_window}: {avg:.1f} >= {early_stop_reward} "
                    f"→ saved '{ckpt_path}'"
                )
                break

        for ms_step, pct in sorted(milestones.items()):
            if step >= ms_step and pct not in milestones_done:
                milestones_done.add(pct)
                n      = min(100, len(reward_history))
                avg_ms = np.mean(reward_history[-n:]) if reward_history else 0.0
                ckpt_path = f"{ckpt_base}_ckpt{pct}pct.pth"

                if avg_ms > last_ckpt_avg:
                    torch.save(policy.state_dict(), ckpt_path)
                    pbar.write(
                        f"[{pct:>3d}%] step {step:>7,} | ep {episode:>5,} "
                        f"| avg100: {avg_ms:.2f} ↑ {last_ckpt_avg:.2f} → saved '{ckpt_path}'"
                    )
                    last_ckpt_avg  = avg_ms
                    last_ckpt_path = ckpt_path
                elif last_ckpt_path is not None and avg_ms < last_ckpt_avg * rollback_threshold:
                    policy.load_state_dict(torch.load(last_ckpt_path, map_location=device, weights_only=True))
                    opt    = optim.Adam(policy.parameters(), lr=effective_lr)
                    lstm_h = torch.zeros(n_envs, LSTM_HIDDEN, device=device)
                    lstm_c = torch.zeros(n_envs, LSTM_HIDDEN, device=device)
                    rollbacks += 1
                    pbar.write(
                        f"[{pct:>3d}%] step {step:>7,} | ep {episode:>5,} "
                        f"| avg100: {avg_ms:.2f} ↓ {last_ckpt_avg:.2f} (threshold {last_ckpt_avg * rollback_threshold:.2f}) → ROLLBACK #{rollbacks}"
                    )
                else:
                    pbar.write(
                        f"[{pct:>3d}%] step {step:>7,} | ep {episode:>5,} "
                        f"| avg100: {avg_ms:.2f} ↓ {last_ckpt_avg:.2f} → within tolerance, continue"
                    )
                    if last_ckpt_path is None:
                        torch.save(policy.state_dict(), ckpt_path)
                        last_ckpt_avg  = avg_ms
                        last_ckpt_path = ckpt_path

        with torch.inference_mode():
            states_t    = torch.tensor(states, dtype=torch.float32).to(device)
            _, last_vals, _ = policy(states_t, (lstm_h, lstm_c))
        last_values = last_vals.squeeze(-1).cpu().numpy()

        ppo_update(policy, opt, buffer, last_values, entropy_coef=entropy_coef, ppo_epochs=ppo_epochs)
        buffer.clear()

    pbar.close()
    vec_env.close()
    torch.save(policy.state_dict(), save_path)
    print(f"Training complete. Model saved to '{save_path}'")
    return reward_history

## 9. Run Training

Available scenarios: `basic`, `defend_the_center`, `deadly_corridor`, …

In [ ]:
reward_history_basic = train(
    scenario="basic",
    save_path="model_basic_multienv.pth",
    buttons=BASIC_BUTTONS,
    n_envs=N_ENVS,
    early_stop_reward=86,
    early_stop_window=100,
)

reward_history_defend = train(
    scenario="defend_the_center",
    save_path="model_defend_multienv.pth",
    buttons=FULL_BUTTONS,
    n_envs=N_ENVS,
    total_steps=3_000_000,
    entropy_coef=0.05,
    ppo_epochs=3,
    early_stop_reward=10,
    early_stop_window=100,
)

## 10. Evaluate Saved Model

In [25]:
def evaluate(model_path, scenario="basic", n_episodes=5, render=False, buttons=FULL_BUTTONS):
    num_actions = len(buttons)
    device      = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    env = DoomEnv(scenario, buttons)
    if render:
        env.game.set_window_visible(True)

    policy = ActorCriticLSTM(num_actions).to(device)
    policy.load_state_dict(torch.load(model_path, map_location=device, weights_only=True))
    policy.eval()

    rewards = []
    for ep in range(n_episodes):
        state      = env.reset()
        lstm_state = policy.init_lstm_state()
        ep_reward  = 0.0
        done       = False

        while not done:
            state_t = torch.tensor(state, dtype=torch.float32).unsqueeze(0).to(device)
            with torch.inference_mode():
                logits, _, lstm_state = policy(state_t, lstm_state)
            action = logits.argmax(dim=-1).item()
            state, reward, done = env.step(action)
            ep_reward += reward

        rewards.append(ep_reward)
        print(f"  Ep {ep+1}/{n_episodes} → reward {ep_reward:.1f}")

    env.close()
    print(f"\nMean: {np.mean(rewards):.1f}  |  Std: {np.std(rewards):.1f}")
    return rewards

## 11. Record Video

Runs `record_video.py` in a subprocess — ViZDoom crashes with an access violation inside Jupyter on Windows, so video capture must happen in a separate process.

In [ ]:
!python record_video.py --scenario basic   --model model_basic_multienv.pth  --output video_basic.mp4   --episodes 3
!python record_video.py --scenario defend  --model model_defend_multienv.pth  --output video_defend.mp4  --episodes 3